# Module 318 — GoodWords Spam Filter Baseline

This notebook implements the **Spam Filter Implementation** section only: load the UCI SMS messages, make word-count features, train Multinomial Naive Bayes, and evaluate held-out predictions. Later GoodWords sections can reuse the in-memory `vectorizer` and `classifier`. No attack logic is included here. Use the `AI` Python environment with pandas, NumPy, and scikit-learn installed.

The classifier sees numeric word counts, not message meaning as a person would understand it.

## Formula the code implements

Bayes' rule is `P(C|D) = P(D|C) P(C) / P(D)`. Read `P(C|D)` aloud as **“probability of class C given document D.”** `C` is spam or ham; `P(C)` is its prior; `P(D|C)` measures how likely the message is under that class. The model selects `arg max`—**“the argument that maximizes”**—the class score.

Naive Bayes approximates `P(D|C)` as the product, from `i=1` to `n`, of `P(w_i|C)`. Say `w_i` as **“w sub i.”** This assumes word features are conditionally independent given the class. Taking logs turns the product into the sum `log P(C) + sum_i log P(w_i|C)`, avoiding multiplication of many tiny probabilities.

With vocabulary `[free, hello]`, the text `free free hello` becomes the count vector `[2, 1]`.

In [ ]:
from io import BytesIO
from pathlib import Path
import csv
import html
import random
import re
import unicodedata
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Fixed seeds make the random train/test split reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_PATH = DATA_DIR / 'sms_spam.csv'
print('Dataset cache:', DATASET_PATH.resolve())

## Load and cache the dataset

The first run downloads the course-linked UCI archive and writes a parsed CSV into `data/`. Reruns use that local cache. The archive is parsed in memory. Size and member-name checks catch an unexpected response before it is treated as training data.

In [ ]:
if DATASET_PATH.is_file():
    df = pd.read_csv(DATASET_PATH)
else:
    dataset_url = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
                   '00228/smsspamcollection.zip')
    with urlopen(dataset_url, timeout=30) as response:
        archive_bytes = response.read()
    if len(archive_bytes) > 5_000_000:
        raise ValueError('Unexpectedly large dataset archive')
    with ZipFile(BytesIO(archive_bytes)) as archive:
        member = archive.getinfo('SMSSpamCollection')
        if member.file_size > 2_000_000:
            raise ValueError('Unexpectedly large SMS file')
        with archive.open(member) as source:
            df = pd.read_csv(source, sep='\t', header=None,
                names=['label', 'message'], quoting=csv.QUOTE_NONE,
                encoding='utf-8')
    df.to_csv(DATASET_PATH, index=False)

assert list(df.columns) == ['label', 'message']
assert df['label'].isin(['ham', 'spam']).all()
print('Rows:', len(df))
print(df['label'].value_counts())
display(df.groupby('label', group_keys=False).head(2)[['label', 'message']])

## Normalize text but preserve useful clues

We normalize encoding, case, and whitespace while retaining numbers, currency, and selected punctuation. Aggressive cleaning can erase spam evidence before the classifier can learn it. A retained character only becomes a model feature if the vectorizer's token rules accept it.

In [ ]:
def minimal_clean(text):
    """Normalize HTML entities, Unicode, and whitespace."""
    text = html.unescape(text)
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    """Lowercase and retain common spam indicators."""
    text = text.lower()
    text = re.sub(r"[^\w\s£$€¥!?.,;:'\"-]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_message'] = df['message'].map(minimal_clean).map(clean_text)
before = len(df)
df = df.drop_duplicates(subset=['label', 'clean_message']).copy()
df = df[df['clean_message'].str.len() > 0].copy()
print('Rows removed:', before - len(df))
print('Rows retained:', len(df))
print('Class proportions (%):')
print((100 * df['label'].value_counts(normalize=True)).round(1))

## Split before fitting the vocabulary

The training data teaches both the vocabulary and class word statistics. The held-out test data measures performance after training. Fitting a vectorizer on all rows would expose vocabulary information from the test set. `stratify=y` keeps the class proportions similar in both groups.

In [ ]:
X_text = df['clean_message']
y = df['label']
train_text, test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=SEED, stratify=y
)
print('Training rows:', len(train_text), 'Test rows:', len(test_text))
print('Train class mix:\n', y_train.value_counts(normalize=True).round(3))
print('Test class mix:\n', y_test.value_counts(normalize=True).round(3))

## Create sparse word-count features

`fit_transform` learns token-to-column mappings from training messages and encodes their counts. `transform` uses that same mapping on test messages. A sparse matrix stores only nonzero entries instead of allocating memory for every absent word in every message.

In [ ]:
vectorizer = CountVectorizer(
    max_features=3000,
    token_pattern=r'\b\w+\b|[£$€¥]+|\d+|!!+|\?\?+|\.\.+',
    lowercase=True, stop_words='english'
)
X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)
feature_names = vectorizer.get_feature_names_out()
print('Train matrix (messages, features):', X_train.shape)
print('Test matrix (messages, features):', X_test.shape)
print('Storage format:', X_train.format)
print('Example vocabulary:', feature_names[:20].tolist())

## Train and evaluate Multinomial Naive Bayes

`fit` estimates class priors and smoothed word-count likelihoods. Accuracy is the share of all predictions that are correct. For spam, precision asks what fraction of messages predicted as spam truly are spam; recall asks what fraction of all actual spam messages the model catches. Because ham is the majority class, accuracy alone can hide poor spam detection.

In [ ]:
classifier = MultinomialNB()
classifier.fit(X_train, y_train)
test_predictions = classifier.predict(X_test)
print('Learned classes:', classifier.classes_.tolist())
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 4))
print('Confusion matrix (actual rows; predicted columns; ham, spam):')
print(confusion_matrix(y_test, test_predictions, labels=['ham', 'spam']))
print(classification_report(y_test, test_predictions,
    labels=['ham', 'spam'], target_names=['ham', 'spam'], zero_division=0))

## Baseline boundary and COAE recall

HTB's displayed accuracy and per-class metrics are run-specific examples; dataset, split, tokenization, and library versions can change them. This notebook learns the vocabulary from training rows only and avoids loading serialized Python pickle files. It stores the fitted `vectorizer` and `classifier` in memory for later sections.

**Remember:** Multinomial Naive Bayes adds word evidence by class; the bag-of-words representation ignores context; stratify the split; evaluate minority-class precision and recall, not accuracy alone. The next section below reuses this fitted model for the GoodWords experiment.

# Section 4 — GoodWords Attack Implementation

HTB assumes **white-box access**: we can inspect the model vocabulary and its learned word probabilities. We will rank words by their relative ham evidence, append the top candidates to held-out spam text, and measure how often the unchanged classifier predicts ham. This section uses the trained `classifier` and `vectorizer` above.

The internal matrix `feature_log_prob_[class_index, feature_index]` stores `log P(w_i | class)`. Read `w_i` as **“w sub i.”** We use `classifier.classes_` to locate the ham and spam rows rather than assuming a fixed class order.

In [ ]:
class_names = classifier.classes_.tolist()
ham_index = class_names.index('ham')
spam_index = class_names.index('spam')

# Each row is a class; each column lines up with one CountVectorizer feature.
ham_log_probs = classifier.feature_log_prob_[ham_index]
spam_log_probs = classifier.feature_log_prob_[spam_index]
assert len(feature_names) == len(ham_log_probs) == len(spam_log_probs)

# exp(log(p)) recovers p. Epsilon prevents a zero denominator in the ratio.
ham_probs = np.exp(ham_log_probs)
spam_probs = np.exp(spam_log_probs)
epsilon = 1e-10
goodness = ham_probs / (spam_probs + epsilon)
goodness_table = pd.DataFrame({
    'word': feature_names,
    'goodness': goodness,
    'ham_probability': ham_probs,
    'spam_probability': spam_probs,
}).sort_values('goodness', ascending=False, ignore_index=True)
top_good_words = goodness_table.head(100).copy()
display(top_good_words.head(10))

## What the goodness score means

The course formula is `S(w) = P(w|ham) / (P(w|spam) + epsilon)`. Say it aloud as **“S of w equals probability of w given ham divided by probability of w given spam plus epsilon.”** A score above one means the estimated word probability is larger for ham than spam. This is a ranking ratio, not a probability. A very rare feature can receive a huge ratio because its spam probability is tiny, so inspect the two absolute probabilities alongside the ratio. Another stable ranking can compare their log-probability difference, but we retain HTB's ratio here to match the lesson.

`epsilon = 1e-10` is ten to the minus ten. It keeps the denominator nonzero when the estimated spam probability is zero. The score can still become very large; epsilon prevents division by zero, it does not make a rare word automatically useful or natural.

## Augment raw held-out spam messages and measure evasion

We keep the original held-out strings in `test_text`. The transformed `X_test` is a sparse count matrix and cannot be concatenated with ordinary words. For each augmented string, the same fitted vectorizer creates a one-row sparse matrix; the classifier returns probabilities in the order shown by `classifier.classes_`.

For binary classes, `P(ham|message) > P(spam|message)` is equivalent to ham probability exceeding 0.5. Evasion rate is `evaded / total spam tested × 100%`. The zero-word point measures the model's baseline false negatives, not evasion caused by added words.

In [ ]:
spam_test_text = test_text.loc[y_test == 'spam']
if spam_test_text.empty:
    raise ValueError('The held-out split contains no spam messages')

word_counts = [0, 5, 10, 15, 20, 25, 30, 35, 40]
attack_results = []
for number_of_words in word_counts:
    selected_words = top_good_words['word'].head(number_of_words).tolist()
    evaded = 0

    for original_message in spam_test_text:
        # Add text first, then vectorize with the original fitted vocabulary.
        if selected_words:
            augmented_message = original_message + ' ' + ' '.join(selected_words)
        else:
            augmented_message = original_message

        one_message_vector = vectorizer.transform([augmented_message])
        probabilities = classifier.predict_proba(one_message_vector)[0]
        if probabilities[ham_index] > probabilities[spam_index]:
            evaded += 1

    total = len(spam_test_text)
    evasion_rate = 100.0 * evaded / total
    attack_results.append({
        'num_words': number_of_words,
        'evaded': evaded,
        'total': total,
        'evasion_rate': evasion_rate,
    })

results_df = pd.DataFrame(attack_results)
display(results_df)

## Plot the measured attack curve

Each row of `results_df` is an observed percentage for one number of appended words. With a finite set of messages, the plot moves in steps; one additional message changes the rate by `100 / total` percentage points. HTB describes its curve as sigmoid-like, but a smooth sigmoid is not guaranteed. These horizontal guides are visual reference points, not universal security thresholds.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df['num_words'], results_df['evasion_rate'],
        marker='o', linewidth=2, color='#00ff9c', label='Measured evasion')
ax.axhline(50, linestyle='--', color='#f2c94c', alpha=0.8, label='50% reference')
ax.axhline(90, linestyle='--', color='#56ccf2', alpha=0.8, label='90% reference')
ax.set(xlabel='Good words appended', ylabel='Evasion rate (%)',
       title='GoodWords attack effectiveness', ylim=(0, 100))
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

### Section 4 takeaways

- White-box access lets the attacker rank learned word statistics directly.
- Exponentiating `feature_log_prob_` recovers probabilities; `classes_` tells us the correct class-row indices.
- Append to raw text, then use the same vectorizer and classifier.
- Compare ham and spam probabilities; record the fraction of held-out spam predicted as ham.
- The notebook's measurements will differ from HTB's displayed 128-message results because the local baseline is freshly trained. Keep that local baseline fixed when comparing attack settings.

Only the Section 4 attack and visualization have been added. Later module sections remain unimplemented until their instructions are reviewed.